In [4]:
import requests
import pandas as pd

url = "https://understat.com/getLeagueData/EPL/2025"
headers = {
    "User-Agent": "Mozilla/5.0",
    "X-Requested-With": "XMLHttpRequest",   # required, otherwise you get an empty page
    "Referer": "https://understat.com/",
}

response = requests.get(url, headers=headers, timeout=30)
print("Status:", response.status_code)      # 200 means success
data = response.json()
print(data.keys())                          # teams, players, dates

Status: 200
dict_keys(['teams', 'players', 'dates'])


In [5]:
rows = []
for team_id, t in data["teams"].items():
    for match in t["history"]:
        rows.append({**match, "team_id": team_id, "team": t["title"]})

matches = pd.json_normalize(rows)
print(matches.shape)    # expect (760, ...) = 20 teams x 38 matches
matches.head()

(760, 23)


,h_a,xG,xGA,npxG,npxGA,deep,deep_allowed,scored,missed,xpts,...,draws,loses,pts,npxGD,team_id,team,ppda.att,ppda.def,ppda_allowed.att,ppda_allowed.def
0,h,0.318601,1.400980,0.318601,1.400980,2,6,0,0,0.4258,...,1,0,1,-1.082379,71,Aston Villa,227,12,146,24
1,a,0.976852,1.129640,0.976852,1.129640,14,2,0,1,1.1287,...,0,1,0,-0.152788,71,Aston Villa,92,17,302,28
2,h,1.205860,2.597130,1.205860,1.835960,6,5,0,3,0.3812,...,0,1,0,-0.630100,71,Aston Villa,222,21,288,15
3,a,0.354327,1.892700,0.354327,1.892700,3,10,0,0,0.3143,...,1,0,1,-1.538373,71,Aston Villa,159,19,191,27
4,a,0.939354,0.938312,0.939354,0.938312,16,4,1,1,1.3300,...,1,0,1,0.001042,71,Aston Villa,112,16,227,23


In [6]:
matches.to_csv("epl_2025_matches.csv", index=False)


| Column | Meaning |
|---|---|
| `h_a` | `h` = home, `a` = away. |
| `date` | Kickoff date and time. |
| `result` | `w` = win, `d` = draw, `l` = loss. |
| `wins`, `draws`, `loses` | Binary flags (`1` or `0`) for the outcome of that match, not running totals. |
| `pts` | Points earned: 3 for a win, 1 for a draw, 0 for a loss. |
| `scored` | Goals scored by the team. |
| `missed` | Goals conceded by the team. |
| `xG` | Expected goals: the combined quality of the team's shots. |
| `xGA` | Expected goals against: the quality of chances allowed. |
| `npxG` | Expected goals excluding penalties. |
| `npxGA` | Expected goals against excluding penalties. |
| `npxGD` | Non-penalty expected goal difference: `npxG - npxGA`. |
| `xpts` | Expected points based on the quality of chances created and conceded. |
| `deep` | Completed passes within approximately 20 metres of the opponent's goal. |
| `deep_allowed` | Completed passes by the opponent within approximately 20 metres of this team's goal. |
| `ppda.att`, `ppda.def` | PPDA components: opponent passes and this team's defensive actions. `ppda.att / ppda.def` gives PPDA; lower values generally indicate more aggressive pressing. |
| `ppda_allowed.att`, `ppda_allowed.def` | PPDA components describing the opponent's pressing against this team. |
| `team_id`, `team` | Understat's team ID and name, added during data collection. |

